In [67]:
!pip install langchain langchain-text-splitters langchain-community bs4
!pip install -U langchain-mistralai

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


### Habilitando o LangSmith (Opcional)

O Langsmith é uma ferramenta que permite a observabilidade nos nossos sistemas de agentes

- Crie uma conta no langsmith: https://smith.langchain.com/
- Na página principal, após o login, clique em "settings". Depois "API Keys"
- Por fim clique em "+ API Key" e crie uma nova chave do tipo "Personal Access Token"

In [68]:
import os
from dotenv import load_dotenv
load_dotenv("/workspaces/ml-supervised-dev/.env")

langsmith_key = os.getenv("LANGSMITH_API_KEY")
if not langsmith_key:
    print("API KEY não fornecida")

### Escolhendo o modelo de chat

Estamos usando o Mistral como provedor e o modelo "mistral-small-latest".

É necessário ter uma variável de ambiente (MISTRAL_API_KEY) com uma API Key configurada

In [69]:
'''
Cria objeto para consumir modelos do Mistral. Outros provedores são bem parecidos.
Link da documentação: https://python.langchain.com/api_reference/mistralai/chat_models/langchain_mistralai.chat_models.ChatMistralAI.html#langchain_mistralai.chat_models.ChatMistralAI.get_num_tokens_from_messages
'''

import os
from langchain.chat_models import init_chat_model

model = init_chat_model("mistral-small-latest")

In [70]:
model

ChatMistralAI(client=<httpx.Client object at 0x7ad0dc323650>, async_client=<httpx.AsyncClient object at 0x7ad0dc323470>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model='mistral-small-latest', model_kwargs={})

### Definindo o modelo de embeddings

Aqui escolhemos o modelo que será usado para "vetorizar" a query do usuário e as informações em nossa base de dados

In [71]:
from langchain_mistralai import MistralAIEmbeddings

embeddings = MistralAIEmbeddings(model="mistral-embed")

In [72]:
embeddings

MistralAIEmbeddings(client=<httpx.Client object at 0x7ad0dc323b90>, async_client=<httpx.AsyncClient object at 0x7ad0dc31b050>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1/', max_retries=5, timeout=120, wait_time=30, max_concurrent_requests=64, tokenizer=Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<unk>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"<s>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"</s>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=Sequence(normalizers=[Prepend(prepend="▁"), Replace(pattern=String(" "), content="▁")]), pre_tokenizer=None, post_processor=TemplateProcessing(single=[SpecialToken(id="<s>", type_id=0), Sequence(id=A, type_id=0)], pair=[SpecialToken(id="<s>", type_id=0), Sequence(id=A, type_id=0)

### Definindo o método de armazenamento de vetores (Vector Store)

Neste exemplo, por motivos de simplicidade, vamos salvar os vetores em memória.

In [73]:
!pip install -U "langchain-core"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [74]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

### Indexing

Nesta etapa buscamos as informações complementares para o nosso modelo

In [75]:
'''
Documento loaders. Objetos que nos permitem acessar
e carregas os "documents" onde estão as informações que queremos
'''

import bs4 # Ferramenta para scraping de páginas web.
from langchain_community.document_loaders import WebBaseLoader # Carrega documentos de páginas web.

loader = WebBaseLoader(
    web_paths=("https://microservices.io/patterns/data/database-per-service.html","https://microservices.io/patterns/reliability/circuit-breaker.html",)
)
docs = loader.load()

for doc in docs:
    doc.page_content = doc.page_content.replace("\n", " ")

assert len(docs) == 2
print(f"Total characters: {len(docs[0].page_content) + len(docs[1].page_content)}")

Total characters: 16873


In [76]:
print(docs[0].page_content[:500])

      Pattern: Database per service                                     Microservice Architecture Supported by Kong     Patterns Articles Presentations Adoptnew Refactoringnew Testingnew   About        Pattern: Database per service        pattern           application architecture           loose coupling      Context Let’s imagine you are developing an online store application using the Microservice architecture pattern. Most services need to persist data in some kind of database. For example, 


### Spliting

O texto completo pode ser maior que o contexto do nosso modelo. Por esse motivo é necessário dividir em "chunks" para gerar os vetores

In [77]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 21 sub-documents.


In [78]:
all_splits[0]

Document(metadata={'source': 'https://microservices.io/patterns/data/database-per-service.html', 'title': 'Pattern: Database per service', 'language': 'No language found.', 'start_index': 6}, page_content='Pattern: Database per service                                     Microservice Architecture Supported by Kong     Patterns Articles Presentations Adoptnew Refactoringnew Testingnew   About        Pattern: Database per service        pattern \xa0         application architecture \xa0         loose coupling \xa0    Context Let’s imagine you are developing an online store application using the Microservice architecture pattern. Most services need to persist data in some kind of database. For example, the Order Service stores information about orders and the Customer Service stores information about customers.  Problem What’s the database architecture in a microservices application? Forces   Services must be loosely coupled so that they can be developed, deployed and scaled independently

### Storing

Armazenamos os chunks em memória, já convertidos em vetores. Idealmente **não salvamos em memória mas sim em bancos de dados vetoriais**

In [79]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['2f6549ad-4877-4dd9-a201-f0a272663c4e', '49c7c287-673c-4f21-9ad4-c8716abd6d3d', 'f95df48e-06b7-4d19-8991-41f2ed6fd5fd']


### Retrieval

Criamos uma tool do tipo code-agent que irá executar o processo de recuperação da informação em nossa base vetorial

In [80]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

### Agente

In [81]:
from langchain.agents import create_agent


tools = [retrieve_context]

prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [82]:
query = (
    "How would you explain database per service pattern?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream( # Streaming response. Permite ver a resposta parcial enquanto é gerada
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

How would you explain database per service pattern?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (a8n9cQMxp)
 Call ID: a8n9cQMxp
  Args:
    query: database per service pattern
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://microservices.io/patterns/data/database-per-service.html', 'title': 'Pattern: Database per service', 'language': 'No language found.', 'start_index': 6}
Content: Pattern: Database per service                                     Microservice Architecture Supported by Kong     Patterns Articles Presentations Adoptnew Refactoringnew Testingnew   About        Pattern: Database per service        pattern           application architecture           loose coupling      Conte